# 14.5 - Serving & Cost Optimization

**Module 14 - LLMOps** | Netsetos GenAI Engineering

This notebook is a runnable cost-optimization playbook: eight tiny models that each price one lever for cutting a runaway LLM bill. You aggregate per-feature spend from traces, then apply prompt caching, contrast it with the KV cache, gate a model route, walk the quantization cliff, price the batch discount, find the self-host crossover, and finally stack the levers to see them compound.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This notebook is deliberately dependency-light: every cost model is plain Python arithmetic, so nothing needs to be installed to run the cells below. The one production example (the real Anthropic cached call) is illustrative and never executed, so its SDK is optional.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic openai -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

Pure environment prep. The install line is commented out because the runnable cells use only the standard library; only the one illustrative production example would need the `anthropic` and `openai` SDKs, and it is not run.

**How the code works - step by step**
- **Commented `pip install anthropic openai`** - uncomment only if you want to run the illustrative production caching example; the eight numbered cost models need nothing.
- **Reproducibility note** - flags that the lesson uses fixed, seeded values throughout, so every printed number is deterministic.

**In one line:** nothing to install - the cost models are just arithmetic.

**What you'll see:** (no output - environment prep)

## 1 - Measure first: find the expensive slice

Before touching a single prompt, aggregate per-feature spend from the traces you already emit (14.1) and find the Pareto. Almost always a small slice of calls drives most of the bill, and it is rarely the one you would have guessed. Optimising the wrong slice is the classic wasted afternoon, so the whole playbook starts here.

In [ ]:
# You cannot cut what you cannot see. Before optimising, aggregate the per-feature cost from your traces (Lesson 14.1)
# and find the Pareto: a small slice of calls usually drives most of the bill. Optimise THAT, not the cheap stuff.
traces = [("classify", 120, 0.0003), ("rag_answer", 50, 0.030), ("summarize", 30, 0.004)]  # (feature, calls, $/call) illustrative
total = sum(calls * cost for _, calls, cost in traces)
total_calls = sum(calls for _, calls, cost in traces)
print("Per-feature cost (from the trace log):")
for feat, calls, cost in sorted(traces, key=lambda t: -t[1] * t[2]):
    spend = calls * cost
    print("  {:<12} {:>4} calls  ${:.4f}/call  -> ${:.4f}  ({:.0%} of bill, {:.0%} of calls)".format(
          feat, calls, cost, spend, spend / total, calls / total_calls))
top = max(traces, key=lambda t: t[1] * t[2])
print()
print("The {} feature is {:.0%} of the bill from just {:.0%} of the calls - that is where the money is.".format(
      top[0], top[1] * top[2] / total, top[1] / total_calls))
print("Optimise the expensive slice; shaving the cheap classify calls would save almost nothing. Measure first.")

# Output:
# Per-feature cost (from the trace log):
#   rag_answer     50 calls  $0.0300/call  -> $1.5000  (91% of bill, 25% of calls)
#   summarize      30 calls  $0.0040/call  -> $0.1200  (7% of bill, 15% of calls)
#   classify      120 calls  $0.0003/call  -> $0.0360  (2% of bill, 60% of calls)
#
# The rag_answer feature is 91% of the bill from just 25% of the calls - that is where the money is.
# Optimise the expensive slice; shaving the cheap classify calls would save almost nothing. Measure first.

**Explanation**

A measurement harness, not a model call. It takes a tiny trace log of `(feature, calls, cost-per-call)` tuples and turns it into a per-feature spend breakdown, then names the single feature that dominates the bill. The point it drives home: cost is `calls x cost-per-call`, and those two axes rarely agree.

**How the code works - step by step**
- **`traces`** - three illustrative features with their call counts and dollar-per-call, standing in for aggregated trace data from Lesson 14.1.
- **`total` / `total_calls`** - sum the whole bill and the whole call volume so each feature can be shown as a share of both.
- **The sorted loop** - orders features by spend (`calls x cost`) descending and prints each one's share of the bill against its share of the calls, exposing the mismatch.
- **`top`** - picks the single highest-spend feature and states it is most of the bill from a fraction of the calls.

**In one line:** aggregate calls x cost per feature, sort by spend, attack the slice that is most of the bill.

**What you'll see:** A per-feature table showing `rag_answer` at 91% of the bill from just 25% of the calls, `summarize` at 7%, and `classify` at 2% despite being 60% of the calls - then a line naming `rag_answer` as where the money is.

## 2 - Prompt caching: pay full price only for what changes

The cheapest lever touches no model choice at all. A request is a large stable prefix (system + few-shot + retrieved docs) plus a small variable suffix (the user query); provider caching bills that prefix at a deep discount after the first call. The one rule that makes it work: keep the variable part last so the prefix stays byte-identical.

In [ ]:
# PROMPT CACHING: a request is a stable PREFIX (system + few-shot + retrieved docs) plus a variable SUFFIX (the user
# query). Provider caching bills the prefix at a deep discount after the first call - so put the variable part LAST.
PRICE_IN = 3.00          # $/1M normal input tokens (illustrative)
CACHE_READ = 0.30        # Anthropic: cache reads = 10% of input (a 90% discount)
CACHE_WRITE = 3.75       # Anthropic: cache writes = 1.25x input (paid once, on the first call)
prefix, suffix = 1800, 60   # tokens: the stable prefix vs the variable user query
no_cache = (prefix + suffix) * PRICE_IN / 1e6
first_call = (prefix * CACHE_WRITE + suffix * PRICE_IN) / 1e6   # writes the cache
warm_call = (prefix * CACHE_READ + suffix * PRICE_IN) / 1e6     # reads the cache
K = 100
no_cache_total = K * no_cache
cache_total = first_call + (K - 1) * warm_call
print("Per call: no-cache ${:.6f}   first(write) ${:.6f}   warm(read) ${:.6f}".format(no_cache, first_call, warm_call))
print("A warm call is {:.0%} of a no-cache call - about {:.0%} off, because the prefix drops to 10% of input.".format(warm_call / no_cache, 1 - warm_call / no_cache))
print("Over {} calls: no-cache ${:.4f}  vs  cached ${:.4f}  ->  {:.0%} cheaper.".format(K, no_cache_total, cache_total, 1 - cache_total / no_cache_total))
print()
print("Cache the stable prefix, pay full price only for the suffix. Change the prefix and you bust the cache - keep the")
print("variable part LAST. (Anthropic reads = 10% of input / writes 1.25x; OpenAI auto-caching = 50% off the cached prefix.)")

# Output:
# Per call: no-cache $0.005580   first(write) $0.006930   warm(read) $0.000720
# A warm call is 13% of a no-cache call - about 87% off, because the prefix drops to 10% of input.
# Over 100 calls: no-cache $0.5580  vs  cached $0.0782  ->  86% cheaper.
#
# Cache the stable prefix, pay full price only for the suffix. Change the prefix and you bust the cache - keep the
# variable part LAST. (Anthropic reads = 10% of input / writes 1.25x; OpenAI auto-caching = 50% off the cached prefix.)

**Explanation**

A pricing model that contrasts three per-call costs - no cache, first call (writes the cache), and warm call (reads it) - then totals a hundred calls under a warm stream. It quantifies the Anthropic economics: reads at 10% of input, a one-time 1.25x write premium, so the warm run collapses toward the suffix-only cost.

**How the code works - step by step**
- **`PRICE_IN`, `CACHE_READ`, `CACHE_WRITE`** - the three Anthropic rates: normal input, cache read (10% of input), cache write (1.25x, paid once).
- **`prefix`, `suffix`** - 1800 stable tokens vs a 60-token variable query, the shape that makes caching pay.
- **`no_cache` / `first_call` / `warm_call`** - price a call with no cache, the first call that writes the cache, and every warm call that reads it.
- **`K`, `no_cache_total`, `cache_total`** - total 100 calls: one write plus 99 warm reads vs 100 full-price calls.

**In one line:** cache the prefix once, then pay ~10% for it on every later call - keep the variable part last or you bust it.

**What you'll see:** Per-call costs (no-cache $0.005580, warm read $0.000720) showing a warm call is 13% of a cold one (~87% off), and over 100 calls $0.5580 drops to $0.0782 - about 86% cheaper.

## 3 - Prompt caching in production (illustrative)

The same idea as the previous cell, but as the real Anthropic call it models. This is the minimal shape of a cached request: a stable system prefix marked cacheable, the user query kept last, and the usage fields that tell you whether you wrote or read the cache.

> **Needs an Anthropic API key** (set ANTHROPIC_API_KEY) - illustrative, not run here.

In [ ]:
# PRODUCTION PROMPT CACHING - an illustrative minimal example (needs the Anthropic SDK + a key; not run here).

# The STABLE prefix: system instructions + few-shot + retrieved policy docs (~1800 tokens, identical every call).
SYSTEM_PREFIX = "You are a support agent. Policy: ...<about 1800 tokens of stable instructions + examples>..."

def answer(user_query):
    import os
    from anthropic import Anthropic
    client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"], timeout=30)
    resp = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=400,
        system=[{"type": "text", "text": SYSTEM_PREFIX,
                 "cache_control": {"type": "ephemeral"}}],    # mark the prefix cacheable
        messages=[{"role": "user", "content": user_query}])   # the VARIABLE part stays LAST, never cached
    u = resp.usage
    # First call WRITES the cache (u.cache_creation_input_tokens, billed at 1.25x once); every warm call
    # READS it (u.cache_read_input_tokens, billed at 10% of input). Change the prefix and you bust the cache.
    return resp.content[0].text, u.cache_creation_input_tokens, u.cache_read_input_tokens
# Output: (illustrative - needs the Anthropic SDK + ANTHROPIC_API_KEY; a real cached call, not run here.)

**Explanation**

A reference snippet, not an executed cell. It shows exactly where the `cache_control` marker goes and which `usage` counters distinguish a cache write from a cache read, so you can wire caching into a real support-agent call.

**How the code works - step by step**
- **`SYSTEM_PREFIX`** - the ~1800-token stable block (instructions + few-shot + policy docs) that stays identical every call.
- **`answer(user_query)`** - builds an `Anthropic` client and calls `messages.create`.
- **`system=[{... "cache_control": {"type": "ephemeral"}}]`** - marks the prefix cacheable; `ephemeral` defaults to a ~5-minute window refreshed on each hit.
- **`messages=[{"role": "user", ...}]`** - the variable query kept last so it never enters the cached prefix.
- **`u.cache_creation_input_tokens` vs `u.cache_read_input_tokens`** - the first call writes (billed 1.25x once), warm calls read (billed at 10%).

**In one line:** mark the prefix `ephemeral`, keep the query last, and read the usage counters to confirm write-vs-read.

**What you'll see:** No output - it defines a function and never calls it; it needs the Anthropic SDK and a key, and is shown purely as the production shape of the previous cell's model.

## 4 - KV cache vs prompt cache: the classic mixup

The single most common interview stumble in this area. These are two different caches on two different axes: the KV cache reuses attention state within one request's decode (a latency win, on the GPU, never billed), while the prompt cache reuses a prefix across requests (a cost win, billed at a discount). You want both; do not confuse them.

In [ ]:
# KV CACHE vs PROMPT CACHE - the #1 mixup. They are DIFFERENT caches on DIFFERENT axes. KV cache reuses attention state
# WITHIN one request's decode (a latency win, per-request, on the GPU, not billed). Prompt cache reuses a prefix ACROSS
# requests (a cost win, billed at a discount). You want BOTH; they do not overlap.
n = 200                                   # output tokens generated in one request
naive_attn = n * (n + 1) // 2             # recompute attention over ALL prior tokens at every step (O(n^2))
kv_attn = n                               # KV cache: reuse the cached keys/values, one step of work per token (O(n))
print("WITHIN one request (KV cache), generating {} tokens:".format(n))
print("  naive recompute: {:>6} attention-steps   KV cache: {:>4} steps   -> about {:.0f}x less decode work".format(naive_attn, kv_attn, naive_attn / kv_attn))
print("  scope: one request | effect: faster decode (latency) | lives: on the GPU | billed: no")
print()
print("ACROSS requests (prompt cache), a shared 1800-token prefix reused by many users:")
print("  effect: the input is billed at a discount (10% of input on Anthropic) | scope: many requests | billed: yes, discounted")
print()
print("So: KV cache saves TIME within a request; prompt cache saves MONEY across requests. Different caches, both worth having.")

# Output:
# WITHIN one request (KV cache), generating 200 tokens:
#   naive recompute:  20100 attention-steps   KV cache:  200 steps   -> about 100x less decode work
#   scope: one request | effect: faster decode (latency) | lives: on the GPU | billed: no
#
# ACROSS requests (prompt cache), a shared 1800-token prefix reused by many users:
#   effect: the input is billed at a discount (10% of input on Anthropic) | scope: many requests | billed: yes, discounted
#
# So: KV cache saves TIME within a request; prompt cache saves MONEY across requests. Different caches, both worth having.

**Explanation**

A contrast harness that quantifies the KV cache's within-request win and then states the prompt cache's across-request win alongside it, holding the two apart on every axis - scope, effect, where it lives, whether it is billed.

**How the code works - step by step**
- **`n`** - 200 output tokens generated in one request.
- **`naive_attn = n*(n+1)//2`** - recomputing attention over all prior tokens at every step is O(n^2).
- **`kv_attn = n`** - the KV cache reuses cached keys/values, one step per token, O(n).
- **The ratio print** - naive vs KV steps, showing ~100x less decode work, plus the KV axes: one request, latency, on the GPU, not billed.
- **The prompt-cache print** - restates the other cache's axes: across requests, cost, billed at a discount.

**In one line:** KV cache saves TIME within a request; prompt cache saves MONEY across requests.

**What you'll see:** For 200 tokens: naive 20100 attention-steps vs KV cache 200 (~100x less decode work), each cache tagged with its axes - KV is latency/GPU/not-billed, prompt cache is cost/across-requests/billed-discounted.

## 5 - Model routing, gated on quality

Send easy queries to a cheap model and hard ones to a strong model, and the blended cost drops sharply. But this is the first lever that can cost quality - a cheap model on a query it cannot handle returns a confident worse answer, not an error. So routing passes through the eval gate from 14.3: the cheap route must clear the golden-set bar or it is blocked.

In [ ]:
# MODEL ROUTING: send easy queries to a cheap model and hard ones to a strong model. It cuts the blended cost - but it
# is a QUALITY decision: the cheap route must clear the eval gate (Lesson 14.3) or you have traded cost for silent quality loss.
easy, hard = 700, 300                     # of 1000 queries
OUT = 200                                 # output tokens per query (illustrative; output dominates cost)
CHEAP_OUT, STRONG_OUT = 4.00, 15.00       # $/1M output for a cheap vs a strong model
all_strong = (easy + hard) * OUT * STRONG_OUT / 1e6
routed = (easy * OUT * CHEAP_OUT + hard * OUT * STRONG_OUT) / 1e6
print("All-strong: ${:.4f}    Routed (easy->cheap, hard->strong): ${:.4f}  ->  {:.0%} cheaper.".format(all_strong, routed, 1 - routed / all_strong))
BAR = 0.92
cheap_route_passrate = 0.93               # measured on the golden set for the easy slice
print()
print("But routing is only valid if the cheap route CLEARS THE GATE. Cheap-route pass-rate on the golden set: {:.0%}.".format(cheap_route_passrate))
print("  {:.0%} >= the {:.0%} bar -> route is safe.  (A cheap route at 0.88 < 0.92 would trade the {:.0%} saving for silent quality loss.)".format(
      cheap_route_passrate, BAR, 1 - routed / all_strong))
print("Route on a difficulty signal, measure PER-ROUTE quality, and gate it. (Routing mechanics: Lesson 13.3.)")

# Output:
# All-strong: $3.0000    Routed (easy->cheap, hard->strong): $1.4600  ->  51% cheaper.
#
# But routing is only valid if the cheap route CLEARS THE GATE. Cheap-route pass-rate on the golden set: 93%.
#   93% >= the 92% bar -> route is safe.  (A cheap route at 0.88 < 0.92 would trade the 51% saving for silent quality loss.)
# Route on a difficulty signal, measure PER-ROUTE quality, and gate it. (Routing mechanics: Lesson 13.3.)

**Explanation**

A cost model with a quality gate bolted on. It prices all-strong against a routed split to show the saving, then checks the cheap route's measured golden-set pass rate against a bar - making the ship/no-ship decision explicit rather than purely financial.

**How the code works - step by step**
- **`easy, hard = 700, 300`** - the difficulty split of 1000 queries.
- **`CHEAP_OUT, STRONG_OUT`** - $/1M output for a cheap vs strong model (output dominates cost here).
- **`all_strong` vs `routed`** - everything to the strong model vs easy-to-cheap, hard-to-strong, with the percentage saved.
- **`BAR = 0.92`, `cheap_route_passrate = 0.93`** - the quality bar and the cheap route's measured golden-set score.
- **The gate print** - 0.93 >= 0.92 so the route is safe; it notes a 0.88 route would be blocked despite the saving.

**In one line:** route on difficulty, measure per-route quality, and only ship the cheap lane if it clears the gate.

**What you'll see:** All-strong $3.0000 vs routed $1.4600 (51% cheaper), then the gate check: the cheap route at 93% clears the 92% bar and is safe, with a note that 88% would trade the saving for silent quality loss.

## 6 - Quantization and the quality cliff

Quantization shrinks an open model - VRAM equals params times bits over eight, so four-bit is about four times smaller than sixteen-bit - which lets it run on a cheaper GPU. But quality does not degrade smoothly: it holds nearly flat and then falls off a cliff at low bit-widths. Gate it on the same golden set and pick the smallest quant that still clears the bar.

In [ ]:
# QUANTIZATION shrinks an open model (VRAM = params x bits / 8) - roughly 4x smaller at 4-bit - but quality has a CLIFF.
# Do not eyeball it: measure quality on the golden set (Lesson 14.3) and pick the SMALLEST quant that still clears the bar.
PARAMS_B = 7                              # a 7B open model
QUANTS = [("FP16", 16, 1.00), ("Q8_0", 8, 0.99), ("Q4_K_M", 4, 0.96), ("Q3_K", 3, 0.88), ("Q2_K", 2, 0.71)]  # illustrative quality
BAR = 0.92
print("quant     bits   VRAM      golden-set quality   clears {:.0%} bar?".format(BAR))
chosen = None
for name, bits, quality in QUANTS:
    vram = PARAMS_B * bits / 8
    ok = quality >= BAR
    if ok:
        chosen = (name, vram)
    print("  {:<7}  {:>2}    {:>4.1f} GB   {:.0%}                 {}".format(name, bits, vram, quality, "yes" if ok else "NO (below the cliff)"))
fp16_vram = PARAMS_B * 16 / 8
print()
print("The smallest quant that still clears the bar is {} at {:.1f} GB - about {:.0f}x less VRAM than FP16.".format(chosen[0], chosen[1], fp16_vram / chosen[1]))
print("Below it (Q3, Q2) quality falls off a cliff: you save VRAM but lose the product. Q4_K_M ~96% is the usual safe floor.")
print("Quantization is a QUALITY decision too - gate it on the golden set, exactly like routing. (Mechanics: Lesson 13.5.)")

# Output:
# quant     bits   VRAM      golden-set quality   clears 92% bar?
#   FP16     16    14.0 GB   100%                 yes
#   Q8_0      8     7.0 GB   99%                 yes
#   Q4_K_M    4     3.5 GB   96%                 yes
#   Q3_K      3     2.6 GB   88%                 NO (below the cliff)
#   Q2_K      2     1.8 GB   71%                 NO (below the cliff)
#
# The smallest quant that still clears the bar is Q4_K_M at 3.5 GB - about 4x less VRAM than FP16.
# Below it (Q3, Q2) quality falls off a cliff: you save VRAM but lose the product. Q4_K_M ~96% is the usual safe floor.
# Quantization is a QUALITY decision too - gate it on the golden set, exactly like routing. (Mechanics: Lesson 13.5.)

**Explanation**

A sweep-and-select harness. It walks a list of quant levels for a 7B model, computes each one's VRAM and checks its golden-set quality against the bar, then picks the smallest quant still above the bar - turning a size decision into a gated quality decision.

**How the code works - step by step**
- **`PARAMS_B = 7`** - a 7-billion-parameter open model.
- **`QUANTS`** - a list of (name, bits, illustrative golden-set quality) from FP16 down to Q2_K.
- **`BAR = 0.92`** - the same quality bar routing used.
- **The loop** - computes `vram = PARAMS_B * bits / 8` for each level, marks whether quality clears the bar, and keeps the smallest passing one in `chosen`.
- **The summary** - names Q4_K_M as the smallest quant above the bar (~4x less VRAM than FP16) and flags Q3/Q2 as below the cliff.

**In one line:** compute VRAM per quant, gate each on the golden set, ship the smallest that clears the bar.

**What you'll see:** A quant table: FP16 (14 GB, 100%), Q8_0 (7 GB, 99%), Q4_K_M (3.5 GB, 96%) all pass; Q3_K (88%) and Q2_K (71%) fall below the cliff - and Q4_K_M is chosen as the safe floor at ~4x less VRAM.

## 7 - The batch API: half price if it can wait

Work nobody is waiting on - backfills, bulk summaries, even running your eval suite - returns within a window (up to 24 hours) at about half the realtime price on both Anthropic and OpenAI. The only question is whether a user is watching. And it stacks: batch on top of a warm cache read reaches roughly 95% off the repeated portion.

In [ ]:
# BATCH API: work that can WAIT (returns within a window, up to 24h) is billed at ~50% off on both Anthropic and OpenAI.
# Use it for offline jobs - backfills, summaries, even running your eval suite - and stack it with cache reads.
n_jobs = 50_000                          # offline summaries to backfill
IN_TOK, OUT_TOK = 500, 150
PRICE_IN, PRICE_OUT = 3.00, 15.00        # $/1M
realtime_each = (IN_TOK * PRICE_IN + OUT_TOK * PRICE_OUT) / 1e6
realtime_total = n_jobs * realtime_each
batch_total = realtime_total * 0.50      # batch = 50% off
print("Backfilling {:,} summaries:".format(n_jobs))
print("  realtime: ${:.2f}    batch (50% off, async): ${:.2f}    -> saves ${:.2f}".format(realtime_total, batch_total, realtime_total - batch_total))
print()
print("Batch is for work a USER is not waiting on. Realtime for anything user-facing; batch for backfills, offline")
print("summaries, and running evals. Stack batch (50% off) with a cache read (90% off) -> about 95% off the repeated portion.")

# Output:
# Backfilling 50,000 summaries:
#   realtime: $187.50    batch (50% off, async): $93.75    -> saves $93.75
#
# Batch is for work a USER is not waiting on. Realtime for anything user-facing; batch for backfills, offline
# summaries, and running evals. Stack batch (50% off) with a cache read (90% off) -> about 95% off the repeated portion.

**Explanation**

A straight pricing comparison. It costs a large offline backfill at realtime and at the 50% batch rate, quantifying the discount and reinforcing the one decision rule - is a user waiting? - plus the stacking note.

**How the code works - step by step**
- **`n_jobs = 50_000`** - offline summaries to backfill.
- **`IN_TOK, OUT_TOK`, `PRICE_IN, PRICE_OUT`** - per-job token counts and $/1M input/output rates.
- **`realtime_each` / `realtime_total`** - full-price cost per job and across all 50k jobs.
- **`batch_total = realtime_total * 0.50`** - the same work at the 50% batch discount.
- **The print** - realtime vs batch totals and the dollars saved, plus the rule (user waiting = realtime) and the stacking with cache reads.

**In one line:** if no user is waiting, send it to batch for half price - and stack it with caching.

**What you'll see:** Backfilling 50,000 summaries costs $187.50 realtime vs $93.75 on batch (saves $93.75), with a reminder that batch is for non-user-facing work and stacks with a cache read to ~95% off the repeated portion.

## 8 - The self-host crossover, and why it moves

The biggest decision: at your volume, rent the API or buy the GPU? An API is a variable per-token cost; a GPU is a fixed monthly cost. They cross at break-even volume = GPU cost / API price-per-token - below it the API wins, above it the GPU. The catch people miss: prices move, and a doubled API price halves the crossover.

In [ ]:
# SELF-HOST CROSSOVER: an API is variable per-token; a GPU is a fixed monthly cost. Break-even volume = GPU cost / API
# price-per-token. Utilization decides (Lessons 13.5/13.6). And prices are NOT fixed - a rise MOVES the crossover, so re-decide.
gpu_monthly = 2000.0                      # a rented GPU box, $/month (illustrative)
api_price_per_1m = 3.00                   # $/1M input tokens today
def crossover_tokens(price_per_1m):
    return gpu_monthly / (price_per_1m / 1e6)   # tokens/month where GPU cost == API cost
base = crossover_tokens(api_price_per_1m)
print("At ${:.2f}/1M, self-hosting breaks even at ~{:.0f}M tokens/month; below that the API is cheaper, above it the GPU wins.".format(api_price_per_1m, base / 1e6))
doubled = 6.00                           # a frontier model doubled its price in 2026 - price RISES happen, not just cuts
new = crossover_tokens(doubled)
print("If the vendor DOUBLES the price to ${:.2f}/1M (this happened in 2026), the crossover HALVES to ~{:.0f}M tokens/month".format(doubled, new / 1e6))
print("  -> self-hosting becomes attractive at half the volume. The buy-vs-rent decision is NOT permanent.")
print()
print("Crossover = GPU fixed cost / API price-per-token; utilization decides. But re-run it when prices move (up or down),")
print("and do not hardcode one vendor forever - the ~100x spread from cheap models to the frontier keeps shifting.")

# Output:
# At $3.00/1M, self-hosting breaks even at ~667M tokens/month; below that the API is cheaper, above it the GPU wins.
# If the vendor DOUBLES the price to $6.00/1M (this happened in 2026), the crossover HALVES to ~333M tokens/month
#   -> self-hosting becomes attractive at half the volume. The buy-vs-rent decision is NOT permanent.
#
# Crossover = GPU fixed cost / API price-per-token; utilization decides. But re-run it when prices move (up or down),
# and do not hardcode one vendor forever - the ~100x spread from cheap models to the frontier keeps shifting.

**Explanation**

A break-even calculator that also demonstrates its own instability. It computes the crossover at today's price, then recomputes it at a doubled price to show the line shifting - making the point that the buy-vs-rent decision is not permanent and must be re-run when vendors change prices.

**How the code works - step by step**
- **`gpu_monthly = 2000.0`** - the fixed monthly cost of a rented GPU box.
- **`api_price_per_1m = 3.00`** - today's variable API price per 1M input tokens.
- **`crossover_tokens(price)`** - returns `gpu_monthly / (price / 1e6)`, the tokens/month where GPU cost equals API cost.
- **`base`** - the crossover at $3.00/1M.
- **`doubled = 6.00`, `new`** - a 2026 frontier price doubling and the resulting crossover, showing it halves.

**In one line:** break-even = GPU fixed cost / API price-per-token, and a doubled price halves it - so re-decide when prices move.

**What you'll see:** At $3.00/1M the crossover is ~667M tokens/month; doubling to $6.00/1M halves it to ~333M tokens/month, so self-hosting becomes attractive at half the volume - the decision is not permanent.

## 9 - Stacking the levers on the right slices

The lesson's real claim: no single lever tames a runaway bill - stacking them does. Starting from the measured baseline in cell 1, this applies each lever to the slice it actually fits - cache the expensive prefix, gate-route its easy half, batch the offline slice - and leaves the cheap slice alone, then multiplies the savings out.

In [ ]:
# THE STACK: the levers are not either/or - you STACK them, each on the RIGHT slice from the measure-first audit (block 01).
# Stacking, not any single lever, is what turns a 5x overrun back into a line you control. (Ratios illustrative, per block.)
baseline = 1.50 + 0.12 + 0.036            # the measured monthly bill from block 01: rag_answer + summarize + classify ($)
# 1) Prompt-cache the rag_answer prefix (block 02): its big repeated prefix drops to a warm read -> ~70% off this slice.
rag_cached = 1.50 * 0.30
# 2) Gate-route the easy half of rag_answer to a cheap model (block 04): ~40% off what remains, and it clears the eval bar.
rag_routed = rag_cached * 0.60
# 3) Batch the offline summarize slice (block 06): exactly half price on work nobody is waiting on.
summarize_batched = 0.12 * 0.50
classify = 0.036                          # the cheap slice - block 01 said leave it alone, so we do
stacked = rag_routed + summarize_batched + classify
print("Baseline bill (from block 01):    ${:.3f}".format(baseline))
print("  rag_answer $1.500  -> cache -> ${:.3f}  -> gated route -> ${:.3f}".format(rag_cached, rag_routed))
print("  summarize  $0.120  -> batch -> ${:.3f}".format(summarize_batched))
print("  classify   $0.036  -> left alone (cheap slice, not worth optimising)")
print("Stacked bill:                     ${:.3f}   ->  {:.1f}x cheaper than the baseline.".format(stacked, baseline / stacked))
print()
print("No single lever did this: caching + gated routing + batch, each on the slice it fits, COMPOUND to the multiple -")
print("about {:.1f}x, enough to bring a 5x overrun back to roughly the original plan. Measure, apply, gate, and STACK.".format(baseline / stacked))

# Output:
# Baseline bill (from block 01):    $1.656
#   rag_answer $1.500  -> cache -> $0.450  -> gated route -> $0.270
#   summarize  $0.120  -> batch -> $0.060
#   classify   $0.036  -> left alone (cheap slice, not worth optimising)
# Stacked bill:                     $0.366   ->  4.5x cheaper than the baseline.
#
# No single lever did this: caching + gated routing + batch, each on the slice it fits, COMPOUND to the multiple -
# about 4.5x, enough to bring a 5x overrun back to roughly the original plan. Measure, apply, gate, and STACK.

**Explanation**

A synthesis harness that composes the earlier cells. It reuses the exact baseline numbers from the measure-first audit and chains discount factors slice-by-slice, showing how caching, gated routing, and batch compound to a multiple no single lever reaches.

**How the code works - step by step**
- **`baseline`** - the same $1.656 bill from cell 1 (rag_answer + summarize + classify), not a new number.
- **`rag_cached = 1.50 * 0.30`** - prompt-cache the expensive rag_answer prefix (~70% off that slice).
- **`rag_routed = rag_cached * 0.60`** - gate-route its easy half to a cheap model (~40% more off, and it clears the bar).
- **`summarize_batched = 0.12 * 0.50`** - batch the offline summarize slice at half price.
- **`classify`** - left untouched, because cell 1 said the cheap slice is not worth optimising.
- **`stacked`** - sums the optimised slices and reports the multiple vs baseline.

**In one line:** apply each lever to the slice it fits, leave the cheap slice alone, and let the savings compound.

**What you'll see:** The $1.656 baseline drops to $0.366 - about 4.5x cheaper - as rag_answer goes $1.500 -> cache $0.450 -> gated route $0.270, summarize $0.120 -> batch $0.060, and classify $0.036 is left alone, enough to bring a 5x overrun back to plan.

The through-line is one loop: measure the per-feature spend first, apply the right lever to the expensive slice, measure the delta, and gate anything that touches quality (routing, quantization) on the golden set from 14.3. The final stack cell is the payoff - caching, gated routing, and batch, each on the slice it fits, compound to about 4.5x cheaper, enough to pull a five-times overrun back to plan. Lesson 14.6 ties tracing, evals, and cost into one operated system; platform-scale serving arrives in 17.4.